# 🔢 Computing Omega From First Principles
### EPS Research High-School Exploration Track — Ages 15-18

The EPS Research omega correction is derived from angular velocity:

$$\omega = \frac{V}{R} \quad [\text{rad/Gyr}]$$

The correction formula uses the **gradient** of angular velocity:

$$\omega_{\rm correction} = \left(\frac{V_2}{R_2} - \frac{V_1}{R_1}\right) \left(\frac{R_1}{R_2}\right)^{3/2}$$

where $(R_1, V_1)$ is the innermost point and $(R_2, V_2)$ is the outermost.

**Why does this make sense?**
- $V/R$ is the angular velocity at each point
- The term $(R_1/R_2)^{3/2}$ comes from Kepler's third law scaling

**Reference:** Flynn & Cannaliato (2025), DOI: 10.3389/fspas.2025.1680387

**Prerequisites:** Algebra, understanding of velocity vs angular velocity

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

with open('rotation_curve_corpus_v7.json') as f:
    corpus = json.load(f)

galaxy = next(g for g in corpus['galaxies'] if g['galaxy'] == 'DDO161')
data   = galaxy['data']

R    = np.array([p['Rad']  for p in data])
Vobs = np.array([p['Vobs'] for p in data])

# Step 1: Compute angular velocity at each point
omega_profile = Vobs / R  # rad / (kpc km/s^-1) = ~ rad/Gyr after unit conversion

print("Step 1: Angular velocity profile V/R")
print(f"  {'R (kpc)':>10}  {'V (km/s)':>10}  {'V/R (km/s/kpc)':>16}")
for r, v, w in zip(R[:6], Vobs[:6], omega_profile[:6]):
    print(f"  {r:>10.2f}  {v:>10.2f}  {w:>16.3f}")
print()
print("Notice: V/R decreases with radius — the angular velocity falls outward")
print("This is the key signal the omega correction captures.")

In [ ]:
# Step 2: Compute omega correction from boundary points
R1, V1 = R[0],  Vobs[0]
R2, V2 = R[-1], Vobs[-1]

print(f"Step 2: Boundary points")
print(f"  Inner: R1 = {R1:.2f} kpc,  V1 = {V1:.2f} km/s,  V1/R1 = {V1/R1:.3f}")
print(f"  Outer: R2 = {R2:.2f} kpc,  V2 = {V2:.2f} km/s,  V2/R2 = {V2/R2:.3f}")
print()

omega = (V2/R2 - V1/R1) * (R1/R2)**1.5
print(f"Step 3: Omega correction")
print(f"  ω = (V2/R2 - V1/R1) × (R1/R2)^(3/2)")
print(f"  ω = ({V2/R2:.3f} - {V1/R1:.3f}) × ({R1/R2:.3f})^1.5")
print(f"  ω = {V2/R2 - V1/R1:.3f} × {(R1/R2)**1.5:.3f}")
print(f"  ω = {omega:.3f} km/s/kpc")
print(f"  ω = {omega:.3f} rad/Gyr  (1 km/s/kpc ≈ 1.022 rad/Gyr)")
print()
print(f"Published value: ω = 4.69 rad/Gyr (Flynn & Cannaliato 2025)")

# Apply correction
V_adj = Vobs - R * omega

# Baryonic velocity
Vgas  = np.array([p['Vgas']  for p in data])
Vdisk = np.array([p['Vdisk'] for p in data])
Vbul  = np.array([p['Vbul']  for p in data])
Vbar  = np.where(Vgas < 0, -np.sqrt(Vgas**2+Vdisk**2+Vbul**2),
                             np.sqrt(Vgas**2+Vdisk**2+Vbul**2))

rmse_omega = np.sqrt(np.mean((V_adj - Vbar)**2))
print(f"\nRMSE (corrected vs baryonic): {rmse_omega:.2f} km/s")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.errorbar(R, Vobs, fmt='o', color='#3498db', markersize=5,
            label=r'$V_{\rm obs}$', zorder=5)
ax.plot(R, Vbar,  's-', color='#e74c3c', lw=1.5, label=r'$V_{\rm bar}$ (baryonic)')
ax.plot(R, V_adj, '^-', color='#2ecc71', lw=2,
        label=fr'$V_{{\rm adj}} = V_{{\rm obs}} - R\omega$  ($\omega={omega:.2f}$)')
ax.set_xlabel('Radius R (kpc)', fontsize=12)
ax.set_ylabel('Velocity V (km/s)', fontsize=12)
ax.set_title('DDO161 — Omega Correction Applied\n'
             'Flynn & Cannaliato (2025) | DOI: 10.3389/fspas.2025.1680387', fontsize=10)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('hs_b_02_omega.png', dpi=150, bbox_inches='tight')
plt.show()